<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/09_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 09 — Evaluation

> **Where you are** — you built agents; now you test them (the Testing GenAI course goes much deeper — this is ADK's built-in slice).
> - **You can already:** everything the agent under test uses.
> - **New in this module:** eval files, two scores, and why word-overlap scoring is weak.
> - **First met here:** an *importable Python package* — a folder with an `__init__.py` file. We create one by writing `.py` files from the notebook, because the evaluator must `import` your agent; it cannot read a notebook.

Eight modules of "does the agent work?" answered by eyeballing the event stream. This module is where we stop doing that.

ADK ships an evaluation framework: you describe a conversation as a test case — what the user says, which tools should be called, what answer you expect — and ADK runs your agent, compares, and reports pass/fail.

**What we'll do:**
1. Package the agent so the evaluator can import it.
2. Describe one expected conversation in a `.test.json` file.
3. Run the evaluation — and watch one of the two scores fail on a perfectly good answer.
4. See what to use instead for real work (an LLM as the judge).

**Running cost:** under $0.01.

# Setup

One new thing in the pip line: the `[eval]` extra — it brings in the scoring helpers (and pins litellm; the comments in the cell say why).

In [1]:
# google-adk[eval] brings in scikit-learn, rouge-score, and the eval helpers.
# The eval extra caps litellm below 1.86 (via google-cloud-aiplatform), hence
# the course-wide litellm==1.85.7 pin.
!pip install -q 'google-adk[eval]==2.7.1' litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null
print("✅ Packages installed.")

✅ Packages installed.


## API Key

Same ritual as every module — Colab secret, `.env`, or paste.

In [2]:
import os
OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY: print("✅ API key loaded from .env file.")
    except ImportError: pass
if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
print("✅ Key configured.")

✅ API key loaded from .env file.
✅ Key configured.


## Imports

One import matters today: `AgentEvaluator`. The whole module revolves around it.

In [3]:
import os
import sys, warnings, asyncio, logging, tempfile, json, shutil
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")

import nest_asyncio; nest_asyncio.apply()
os.environ.setdefault("LITELLM_LOG", "ERROR")  # silence LiteLLM's import-time provider warnings
import litellm; litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google.adk.evaluation import AgentEvaluator

print("✅ Imports successful.")

✅ Imports successful.


# The Two Scores ADK Gives You

You tweak an instruction, the agent behaves differently — better? worse? ADK's answer is two scores per test case:

| Score | Asks | Scale | Default threshold |
|---|---|---|---|
| `tool_trajectory_avg_score` | Did the agent call the right tools, in the right order, with the right args? | 0–1 | 1.0 (strict) |
| `response_match_score` | How similar is the actual final answer to the expected one? | 0–1 (ROUGE-1) | 0.8 |

The trajectory score is the unusual one — most frameworks only grade the final output. ADK also grades *how the agent got there*, which is the right idea for tool-heavy agents: a correct answer reached the wrong way is a bug waiting to happen.

Response match uses **ROUGE-1** — word overlap (what fraction of the expected words appear in the actual answer, and vice versa; 0 = no shared words, 1 = identical). That is a crude yardstick for natural language, and in a few cells we'll watch it fail on a perfectly good answer.

# Step 1 — Package the Agent So the Evaluator Can Import It

The evaluator is a separate piece of machinery: it has to `import` your agent, and Python cannot import a notebook. So we write the agent into two small files in a temp folder:

- `__init__.py` — one line; its presence turns the folder into an **importable package**.
- `agent.py` — the same `LlmAgent` you know, assigned to a variable named `root_agent` — the agreed name the evaluator looks for.

It is the same folder layout you saw with `adk web`. The agent itself is our old friend: a weather agent with one `get_weather` tool.

In [4]:
# Set up a temp directory to hold the agent module + eval file.
EVAL_DIR = tempfile.mkdtemp(prefix="adk_m09_")
AGENT_DIR = os.path.join(EVAL_DIR, "eval_demo_agent")
os.makedirs(AGENT_DIR, exist_ok=True)

# __init__.py makes it an importable package. AgentEvaluator resolves the
# agent via `<package>.agent.root_agent`, so the package must re-export the
# `agent` submodule for `hasattr(package, "agent")` to hold.
with open(os.path.join(AGENT_DIR, "__init__.py"), "w") as f:
    f.write("from . import agent\n")

# The agent module itself. `root_agent` is the conventional name the evaluator looks for.
AGENT_CODE = (
    "import os\n"
    "from dotenv import load_dotenv\n"
    "load_dotenv()\n"
    "\n"
    "from google.adk.agents import LlmAgent\n"
    "from google.adk.models.lite_llm import LiteLlm\n"
    "\n"
    "def get_weather(city: str) -> dict:\n"
    "    'Look up today’s weather for a city.'\n"
    "    db = {\n"
    "        'Bratislava': {'city': 'Bratislava', 'condition': 'Sunny', 'temperature_c': 18},\n"
    "        'Prague': {'city': 'Prague', 'condition': 'Cloudy', 'temperature_c': 14},\n"
    "        'Munich': {'city': 'Munich', 'condition': 'Rainy', 'temperature_c': 11},\n"
    "    }\n"
    "    return db.get(city, {'error': f'No data for {city}'})\n"
    "\n"
    "root_agent = LlmAgent(\n"
    "    name='weather_agent',\n"
    "    model=LiteLlm(model='openrouter/openai/gpt-5.6-luna'),\n"
    "    description='Reports weather for European cities.',\n"
    "    instruction=(\n"
    "        'You report weather. Use get_weather to fetch the condition and '\n"
    "        'Celsius temperature. Answer in one sentence.'\n"
    "    ),\n"
    "    tools=[get_weather],\n"
    ")\n"
)

with open(os.path.join(AGENT_DIR, "agent.py"), "w") as f:
    f.write(AGENT_CODE)

# Make the temp dir importable.
if EVAL_DIR not in sys.path:
    sys.path.insert(0, EVAL_DIR)

print(f"✅ Agent module at {AGENT_DIR}")
print(f"   Module path: eval_demo_agent (AgentEvaluator resolves eval_demo_agent.agent.root_agent)")

✅ Agent module at /var/folders/bh/p1vsc7wx553c75bl6f43y79w0000gn/T/adk_m09_auu3c96l/eval_demo_agent
   Module path: eval_demo_agent (AgentEvaluator resolves eval_demo_agent.agent.root_agent)


# Step 2 — Describe the Expected Conversation (`.test.json`)

A test case is a conversation with the surprises removed. For each user turn you declare:

- **`user_content`** — what the user says.
- **`final_response`** — the answer you expect.
- **`intermediate_data.tool_uses`** — which tools you expect called, with which args.
- **`session_input`** — optional starting session state.

In daily work you rarely type this by hand: you chat with the agent in `adk web` and click **"Save as eval"** on a conversation you like — that exports exactly this format. The next cell writes the same file in code, so you can see every field.

In [5]:
EVAL_SET = {
    "eval_set_id": "weather_basic",
    "name": "weather basic",
    "description": "One test case for the weather agent.",
    "eval_cases": [
        {
            "eval_id": "case_prague",
            "conversation": [
                {
                    "invocation_id": "inv-1",
                    "user_content": {
                        "parts": [{"text": "What is the weather in Prague?"}],
                        "role": "user",
                    },
                    "final_response": {
                        "parts": [{"text": "The weather in Prague is cloudy and 14 degrees Celsius."}],
                        "role": "model",
                    },
                    "intermediate_data": {
                        "tool_uses": [
                            {"name": "get_weather", "args": {"city": "Prague"}}
                        ],
                        "intermediate_responses": [],
                    },
                }
            ],
            "session_input": {
                "app_name": "weather_agent",
                "user_id": "tester",
                "state": {},
            },
        }
    ],
}

EVAL_FILE = os.path.join(EVAL_DIR, "weather.test.json")
with open(EVAL_FILE, "w") as f:
    json.dump(EVAL_SET, f, indent=2)

print(f"✅ Eval set written to {EVAL_FILE}")
print(f"   {len(EVAL_SET['eval_cases'])} case(s)")

✅ Eval set written to /var/folders/bh/p1vsc7wx553c75bl6f43y79w0000gn/T/adk_m09_auu3c96l/weather.test.json
   1 case(s)


# Step 3 — Run It (and Watch One Score Fail)

The key line, before you run it:

```python
await AgentEvaluator.evaluate(
    agent_module="eval_demo_agent",
    eval_dataset_file_path_or_dir=EVAL_FILE,
    num_runs=1,
)
```

Three plain answers: *which agent* (the package name we just wrote), *which test file*, *how many times to repeat each case*. It loads both, runs the agent against every case, scores the result, and raises an `AssertionError` if any score lands below its threshold.

One deliberate twist: the cell first sets a **strict** `response_match_score` threshold of 0.95, so you can see exactly how this score behaves.

In [6]:
# Set a strict threshold to make the ROUGE-1 weakness visible.
# Default is 0.8; we bump to 0.95 — almost impossible for free-form text.
STRICT_CONFIG = {
    "criteria": {
        "tool_trajectory_avg_score": 1.0,
        "response_match_score": 0.95,
    }
}
with open(os.path.join(EVAL_DIR, "test_config.json"), "w") as f:
    json.dump(STRICT_CONFIG, f, indent=2)

# Clear the old module from sys.modules so changes are picked up.
for mod in list(sys.modules):
    if mod.startswith("eval_demo_agent"):
        del sys.modules[mod]

try:
    await AgentEvaluator.evaluate(
        agent_module="eval_demo_agent",
        eval_dataset_file_path_or_dir=EVAL_FILE,
        num_runs=1,
    )
    print("\n✅ All eval cases passed.")
except AssertionError as e:
    print(f"\n❌ Eval failed with a strict threshold — see the table above.")

Summary: `EvalStatus.FAILED` for Metric: `response_match_score`. Expected threshold: `0.95`, actual value: `0.4210526315789474`.
+----+-------------------+----------+-------------+------------------------+--------------------------+-------------------------+-----------------------+---------------------------+
|    | eval_status       |    score |   threshold | prompt                 | expected_response        | actual_response         | expected_tool_calls   | actual_tool_calls         |
+====+===================+==========+=============+========================+==========================+=========================+=======================+===========================+
|  0 | EvalStatus.FAILED | 0.421053 |        0.95 | What is the weather in | The weather in Prague is | Prague is cloudy with a | id=None args={'city': | id='call_jlmapPQcLxlMF6Rv |
|    |                   |          |             | Prague?                | cloudy and 14 degrees    | temperature of 14°C.    | 'Prague'}    

### 🔍 What just happened?

Two things, one good and one telling:

1. **Trajectory passed.** The agent called `get_weather({'city': 'Prague'})` — exactly the expected tool, exactly the expected args.
2. **Response match failed.** The agent said *"Prague is cloudy with a temperature of 14°C."*; we expected *"The weather in Prague is cloudy and 14 degrees Celsius."* Word overlap: about 0.42 — far below 0.95, and below the default 0.8 too. **Same information, different words.** A model that answers tersely scores *worse*, which tells you nothing about quality.

This is the one thing to remember from the whole module: **trajectory testing is useful; ROUGE-1 response matching is weak.** It punishes normal variation in phrasing. Treat it as a sanity check, never as a production gate.

### 🎯 Mini-task — break the trajectory on purpose

Edit the agent's instruction in the Step 1 cell to say *"never call get_weather; just make up the weather"*, re-run Step 1 and Step 3. What does `tool_trajectory_avg_score` come back as now? (Undo the sabotage afterwards.)

# Turning the Threshold Down — and What That Admits

Thresholds live in a `test_config.json` file dropped next to your test file:

```json
{
  "criteria": {
    "tool_trajectory_avg_score": 1.0,
    "response_match_score": 0.5
  }
}
```

Plain reading: stay strict on tools, accept any answer sharing half its words with the expected one. In practice 0.6–0.7 works for short answers; for long-form text the score stops meaning anything at all.

The next cell drops the threshold to a very permissive 0.3 and re-runs.

In [7]:
CONFIG = {
    "criteria": {
        "tool_trajectory_avg_score": 1.0,   # stay strict on trajectory
        "response_match_score": 0.3,        # very permissive on text match
    }
}

with open(os.path.join(EVAL_DIR, "test_config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)

# Reload and re-run.
for mod in list(sys.modules):
    if mod.startswith("eval_demo_agent"):
        del sys.modules[mod]

try:
    await AgentEvaluator.evaluate(
        agent_module="eval_demo_agent",
        eval_dataset_file_path_or_dir=EVAL_FILE,
        num_runs=1,
    )
    print("\n✅ All eval cases passed with the permissive threshold.")
except AssertionError as e:
    print(f"\n❌ Still failed:")
    print(str(e)[:1500])


✅ All eval cases passed with the permissive threshold.


### 🔍 What just happened?

The test passes now — but notice what we did. We didn't make the agent better; we made the test easier. Dropping to 0.3 is an admission that ROUGE-1 is the wrong ruler: you gained a green checkmark and lost any real signal about answer quality.

Keep the strict trajectory score. For the answer itself, there is a better ruler — next section.

### 🎯 Mini-tasks

1. **Add a second turn.** Extend the eval case: the user follows up with *"and in Munich?"*, expected tool call `get_weather({"city": "Munich"})`. Does trajectory still pass at 1.0?
2. **Test the failure path.** Add a case for a city not in the database ("Warsaw"): expected trajectory `get_weather("Warsaw")`, expected answer containing "no data". Does the agent handle it?

# The Real Upgrade — Let an LLM Judge the Answer

Production evaluation grades answers with another LLM. The idea in pseudocode:

```python
# Pseudocode — not built into ADK today, but straightforward to wire up
def judge(prompt, expected, actual) -> float:
    judge_llm_response = judge_llm(
        f"Is this response a reasonable answer to this prompt?\n"
        f"Prompt: {prompt}\nExpected: {expected}\nActual: {actual}\n"
        f"Score 0-1 on semantic correctness. Just the number."
    )
    return float(judge_llm_response.strip())
```

A judge model reads meaning, not word overlap — *"14°C and cloudy"* and *"cloudy, 14 degrees"* both score high. ADK 1.29+ ships a Gen AI Evaluation Service integration that does this for Vertex AI users (Public Preview); everyone else wires the judge themselves with one extra LiteLLM call.

The shift to hold on to: trajectory tells you *the agent did the right things*; a judge tells you *it said the right thing*. Both matter; neither is ROUGE-1.

### 🎯 Mini-task — sketch the judge

Sketch a second `LlmAgent` that receives the expected and the actual answer and replies with a 0–1 score. Feed it our failing pair from Step 3. Does it say "reasonable" where ROUGE-1 said 0.42?

# The Everyday Loop — `adk web` + `adk eval`

Everything above ran from Python. Day to day you'd use the command line:

```bash
adk web                                            # chat, then click "Save as eval"
adk eval ./my_agent ./my_agent/my_evals.test.json  # score it; tweak; re-run
```

`adk eval` takes two arguments — the agent folder, the test file — and prints the same scores you saw above. The loop (chat → save → tweak → re-run) is fast enough to run while prototyping. Good habit: the moment the agent gets a query right after some prompt-tuning, save that conversation as an eval case. The test set builds itself as you work.

# One Warning Before You Deploy

**`adk eval` writes files back into the agent's directory** (it persists updated session histories). On a read-only filesystem — common for locked-down Kubernetes containers — it dies with `PermissionError`. Upstream issue: adk-python #3887.

Workaround: make the folder writable during eval runs, or simply run eval outside the deployed container — in CI or on your machine, against the same agent code. Not a classroom problem; it becomes one when you wire eval into CI/CD.

# Cleanup

Remove the temp folder with the agent package and eval files.

In [8]:
# Clean up the temp directory we created.
shutil.rmtree(EVAL_DIR, ignore_errors=True)
print(f"✅ Cleaned up {EVAL_DIR}")

✅ Cleaned up /var/folders/bh/p1vsc7wx553c75bl6f43y79w0000gn/T/adk_m09_auu3c96l


# Key Takeaways

- **Two built-in scores**: `tool_trajectory_avg_score` (strict, genuinely useful) and `response_match_score` (ROUGE-1 word overlap, weak).
- **The trajectory score is what sets ADK's eval apart** — it grades *how* the agent worked, not just what it said. Right idea for tool-heavy agents.
- **ROUGE-1 punishes normal phrasing variation** — a correct answer in different words fails the default 0.8. Sanity check, never a production gate.
- **The real upgrade is an LLM judge** — built into ADK 1.29+ via the Gen AI Evaluation Service for Vertex users, one extra LiteLLM call for everyone else.
- **`.test.json` files come free from `adk web`** — click "Save as eval" on good conversations and the test set builds itself.
- **`adk eval` writes into the agent folder** — read-only deployments need eval to run elsewhere (CI is the natural home).

# Next up — M10: Deployment

The last module of Part 1. `adk deploy cloud_run` as the one-command deploy, a plain Dockerfile that runs the same agent anywhere, and a look at Vertex AI Agent Engine as the managed path. Then Part 2 begins — the Gemini-only features.